In [ ]:
### Schritt 1: Modell trainieren ###
######### "Goldstandard" ###########

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# --- SCHRITT 1: Daten laden & erkunden ---
df = pd.read_csv("Datasets/digits.csv")    # Pfad ggf. anpassen
y  = df["class"]                  # An Datensatz anzupassen
X  = df.drop(columns=["class"])   # An Datensatz anzupassen

# --- SCHRITT 2: Train-Test-Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Pipeline definieren ---
pipeline = Pipeline([
    ('scaler',     StandardScaler()),
    ('pca',        PCA(n_components=0.95, random_state=42)),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000)),
    
])

# --- SCHRITT 4: Kreuzvalidierung ---
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')

print("--- Cross-Validation ---")
print(f"Genauigkeit je Fold: {scores}")
print(f"Durchschnitt:        {np.mean(scores):.2%} (±{np.std(scores):.2%})")

# --- SCHRITT 5: Finales Training ---
pipeline.fit(X_train, y_train)

pca = pipeline.named_steps['pca']
print(f"Originalfeatures: {X_train.shape[1]}")
print(f"PCA-Komponenten:  {pca.n_components_}")
print(f"Erklärte Varianz: {pca.explained_variance_ratio_.sum():.2f}")

# --- SCHRITT 6: Evaluation auf dem Test-Set ---
print(f"Genauigkeit: {pipeline.score(X_test, y_test):.2%}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, pipeline.predict(X_test)))

# --- SCHRITT 7: Modell speichern ---
MODEL_FILE = "Modelle/digits_logReg.pkl"    # An Modell & Datensatz anpassen
joblib.dump(pipeline, MODEL_FILE)
print(f"Modell gespeichert als '{MODEL_FILE}'.")

In [ ]:
### Schritt 2: Deployment starten ###
############# Gradio App ############

import gradio as gr
from PIL import Image
from pathlib import Path

# --- Modell laden ---
pipeline   = joblib.load("Modelle/digits_logReg.pkl")

# --- Vorhersagefunktion ---
def predict(image):
    # Auf 0–16 skalieren (Bilder liegen bereits als 8×8 vor)
    pixel = np.array(Image.fromarray(image).convert("L"), dtype=float) / 255 * 16

    X = pd.DataFrame([pixel.flatten()])

    pred  = pipeline.predict(X)[0]
    proba = pipeline.predict_proba(X)[0]

    return str(pred), {str(i): float(p) for i, p in enumerate(proba)}

# --- Interface ---
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Bild hochladen"),
    outputs=[
        gr.Textbox(label="Erkannte Ziffer"),
        gr.Label(label="Wahrscheinlichkeiten"),
    ],
    title="Ziffernerkennung"
)

demo.launch(inbrowser=True)